# 🚀 OpenCloud-SRE: Autonomous GRPO Training
This notebook contains the complete training pipeline for the OpenCloud-SRE agent using **Unsloth** and **GRPO** (Group Relative Policy Optimization).

### ⚠️ Important: Runtime Setup
To run this notebook, you **MUST** use a GPU runtime:
1. Go to **Runtime** -> **Change runtime type**.
2. Select **T4 GPU** (Standard) or higher.

### 🔗 Links
- **GitHub:** [https://github.com/hitendras510/OpenCloud-SRE](https://github.com/hitendras510/OpenCloud-SRE)
- **Hugging Face:** [hitendras510/OpenCloud-SRE-GRPO](https://huggingface.co/spaces/hitendras510/OpenCloud-SRE)

In [ ]:
# 1. Install Dependencies
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir trl transformers accelerate datasets bitsandbytes wandb matplotlib

In [ ]:
# 2. GPU Verification & Model Loading
import torch
from unsloth import FastLanguageModel

if not torch.cuda.is_available():
    raise RuntimeError("GPU not found! Please switch to a T4 GPU runtime in Colab settings.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print(f"Connected to {torch.cuda.get_device_name(0)} and LoRA initialized!")

In [ ]:
# 3. OpenCloud Environment Simulation
from dataclasses import dataclass

@dataclass
class CloudState:
    traffic: float = 20.0
    temp: float = 30.0
    health: float = 90.0

    def slo_score(self) -> float:
        # Simple combined health metric (0.0 to 1.0)
        dist = ((self.traffic-20)**2 + (self.temp-30)**2 + (self.health-90)**2)**0.5
        return max(0.0, 1.0 - dist / 150.0)

class OpenCloudEnv:
    def reset(self):
        return CloudState(90.0, 85.0, 10.0) # Start in CRITICAL state
    
    def step(self, action):
        # Deterministic deltas for the agent to learn
        if action == "scale_out":
            return CloudState(20.0, 30.0, 95.0), 1.0 # Success
        return CloudState(95.0, 90.0, 5.0), 0.1 # Failure/Noop

In [ ]:
# 4. GRPO Core: Advantage Calculation
def calculate_advantages(rewards):
    r_t = torch.tensor(rewards, dtype=torch.float32)
    std = r_t.std()
    if std < 1e-8:
        return [0.0] * len(rewards)
    return ((r_t - r_t.mean()) / std).tolist()

def build_prompt(state):
    return f"Traffic: {state.traffic}, Temp: {state.temp}, Health: {state.health}. Respond with JSON intent."

In [ ]:
# 5. Training Loop (Single Step Demo)
env = OpenCloudEnv()
state = env.reset()
prompt = build_prompt(state)

print(f"Initial State SLO: {state.slo_score():.2f}")
print("Ready for GRPO Training Iteration.")